<a href="https://colab.research.google.com/github/pleielp/DL-PyTorch/blob/main/%EB%94%A5%EB%9F%AC%EB%8B%9D_%ED%8C%8C%EC%9D%B4%ED%86%A0%EC%B9%98_%EA%B5%90%EA%B3%BC%EC%84%9C/03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03-01

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

torch.manual_seed(1)

In [12]:
# 데이터
x_train = torch.FloatTensor([[1], [2], [3]])
y_train = torch.FloatTensor([[2], [4], [6]])

# 모델 초기화
W = torch.zeros(1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)

# optimizer 설정
optimizer = optim.SGD([W, b], lr=0.01)

nb_epochs = 3000 # 원하는만큼 경사 하강법을 반복
for epoch in range(nb_epochs + 1):

    # H(x) 계산
    hypothesis = x_train * W + b

    # cost 계산
    cost = torch.mean((hypothesis - y_train) ** 2)

    # cost로 H(x) 개선
    optimizer.zero_grad()
    cost.backward()
    optimizer.step()

    # 100번마다 로그 출력
    if epoch % 100 == 0:
        print('Epoch {:4d}/{} W: {:.3f}, b: {:.3f} Cost: {:.6f}'.format(
            epoch, nb_epochs, W.item(), b.item(), cost.item()
        ))

Epoch    0/3000 W: 0.187, b: 0.080 Cost: 18.666666
Epoch  100/3000 W: 1.746, b: 0.578 Cost: 0.048171
Epoch  200/3000 W: 1.800, b: 0.454 Cost: 0.029767
Epoch  300/3000 W: 1.843, b: 0.357 Cost: 0.018394
Epoch  400/3000 W: 1.876, b: 0.281 Cost: 0.011366
Epoch  500/3000 W: 1.903, b: 0.221 Cost: 0.007024
Epoch  600/3000 W: 1.924, b: 0.174 Cost: 0.004340
Epoch  700/3000 W: 1.940, b: 0.136 Cost: 0.002682
Epoch  800/3000 W: 1.953, b: 0.107 Cost: 0.001657
Epoch  900/3000 W: 1.963, b: 0.084 Cost: 0.001024
Epoch 1000/3000 W: 1.971, b: 0.066 Cost: 0.000633
Epoch 1100/3000 W: 1.977, b: 0.052 Cost: 0.000391
Epoch 1200/3000 W: 1.982, b: 0.041 Cost: 0.000242
Epoch 1300/3000 W: 1.986, b: 0.032 Cost: 0.000149
Epoch 1400/3000 W: 1.989, b: 0.025 Cost: 0.000092
Epoch 1500/3000 W: 1.991, b: 0.020 Cost: 0.000057
Epoch 1600/3000 W: 1.993, b: 0.016 Cost: 0.000035
Epoch 1700/3000 W: 1.995, b: 0.012 Cost: 0.000022
Epoch 1800/3000 W: 1.996, b: 0.010 Cost: 0.000013
Epoch 1900/3000 W: 1.997, b: 0.008 Cost: 0.000008

In [10]:
class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = data          # 실제 값
        self.grad = 0.0           # 이 노드에 대한 기울기 (처음엔 0)
        self._backward = lambda: None  # 이 연산을 거꾸로 미분하는 함수
        self._prev = set(_children)    # 이 값을 만든 입력 노드들
        self._op = _op

    def __add__(self, other):
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            # 덧셈의 미분: 양쪽에 그대로 전달 (d(a+b)/da = 1)
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            # 곱셈의 미분: 상대방 값을 곱해 전달 (d(a*b)/da = b)
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def backward(self):
        # 1. 위상정렬: 그래프를 순서대로 나열
        topo = []
        visited = set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build(child)
                topo.append(v)
        build(self)

        # 2. 출발점(자기 자신)의 기울기는 1
        self.grad = 1.0

        # 3. 역순으로 각 노드의 _backward 호출 → 연쇄법칙 전파
        for v in reversed(topo):
            v._backward()

a = Value(2.0)
b = Value(3.0)
c = a * b      # c = 6
d = c + a      # d = 8
d.backward()

print(a.grad)  # 4.0  → dd/da = b + 1 = 3 + 1
print(b.grad)  # 2.0  → dd/db = a = 2

4.0
2.0


# 03-02

In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

torch.manual_seed(1)

In [18]:
# 데이터
x1_train = torch.FloatTensor([[73], [93], [89], [96], [73]])
x2_train = torch.FloatTensor([[80], [88], [91], [98], [66]])
x3_train = torch.FloatTensor([[75], [93], [90], [100], [70]])
y_train = torch.FloatTensor([[152], [185], [180], [196], [142]])

# 모델 초기화
w1 = torch.zeros(1, requires_grad=True)
w2 = torch.zeros(1, requires_grad=True)
w3 = torch.zeros(1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)

# optimizer 설정
optimizer = optim.SGD([w1, w2, w3, b], lr=1e-5)

nb_epochs = 10000 # 원하는만큼 경사 하강법을 반복
for epoch in range(nb_epochs + 1):

    # H(x) 계산
    hypothesis = x1_train * w1 + x2_train * w2 + x3_train * w3 + b

    # cost 계산
    cost = torch.mean((hypothesis - y_train) ** 2)

    # cost로 H(x) 개선
    optimizer.zero_grad()
    cost.backward()
    optimizer.step()

    # 100번마다 로그 출력
    if epoch % 1000 == 0:
        print('Epoch {:4d}/{} w1: {:.3f} w2: {:.3f} w3: {:.3f} b: {:.3f} Cost: {:.6f}'.format(
            epoch, nb_epochs, w1.item(), w2.item(), w3.item(), b.item(), cost.item()
        ))

Epoch    0/10000 w1: 0.294 w2: 0.294 w3: 0.297 b: 0.003 Cost: 29661.800781
Epoch 1000/10000 w1: 0.718 w2: 0.613 w3: 0.680 b: 0.009 Cost: 1.079390
Epoch 2000/10000 w1: 0.757 w2: 0.571 w3: 0.682 b: 0.011 Cost: 0.754379
Epoch 3000/10000 w1: 0.788 w2: 0.541 w3: 0.682 b: 0.012 Cost: 0.562653
Epoch 4000/10000 w1: 0.812 w2: 0.517 w3: 0.681 b: 0.013 Cost: 0.448557
Epoch 5000/10000 w1: 0.832 w2: 0.500 w3: 0.678 b: 0.014 Cost: 0.379739
Epoch 6000/10000 w1: 0.848 w2: 0.488 w3: 0.675 b: 0.015 Cost: 0.337360
Epoch 7000/10000 w1: 0.861 w2: 0.478 w3: 0.671 b: 0.016 Cost: 0.310474
Epoch 8000/10000 w1: 0.871 w2: 0.472 w3: 0.667 b: 0.018 Cost: 0.292709
Epoch 9000/10000 w1: 0.880 w2: 0.467 w3: 0.663 b: 0.019 Cost: 0.280348
Epoch 10000/10000 w1: 0.888 w2: 0.464 w3: 0.658 b: 0.020 Cost: 0.271223


In [27]:
w1, w2, w3, b = 1.05, 0.523, 0.439, 0.107
x1 = [73, 80, 75]
x2 = [93, 88, 93]
x3 = [89, 91, 80]
x4 = [96, 98, 100]
x5 = [73, 66, 70]

def func(x):
  return x[0] * w1 + x[1] * w2 + x[2] * w3 + b

print(func(x1))
print(func(x2))
print(func(x3))
print(func(x4))
print(func(x5))

151.52200000000002
184.608
176.27
196.06100000000004
142.005


In [41]:
# 데이터
x1_train = torch.FloatTensor([[73], [93], [89], [96], [73]])
x2_train = torch.FloatTensor([[80], [88], [91], [98], [66]])
x3_train = torch.FloatTensor([[75], [93], [90], [100], [70]])
y_train = torch.FloatTensor([[152], [185], [180], [196], [142]])

# 정규화: 각 입력을 평균 0, 표준편차 1로
x1_mu, x1_sigma = x1_train.mean(), x1_train.std()
x2_mu, x2_sigma = x2_train.mean(), x2_train.std()
x3_mu, x3_sigma = x3_train.mean(), x3_train.std()

# 데이터 (정규화까지 완료했다고 가정)
x1_norm = (x1_train - x1_train.mean()) / x1_train.std()
x2_norm = (x2_train - x2_train.mean()) / x2_train.std()
x3_norm = (x3_train - x3_train.mean()) / x3_train.std()

# 모델 초기화
w1 = torch.zeros(1, requires_grad=True)
w2 = torch.zeros(1, requires_grad=True)
w3 = torch.zeros(1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)

# weight_decay 추가
optimizer = optim.SGD([w1, w2, w3, b], lr=1e-2, weight_decay=1e-3)

nb_epochs = 5000
for epoch in range(nb_epochs + 1):
    hypothesis = x1_norm * w1 + x2_norm * w2 + x3_norm * w3 + b
    cost = torch.mean((hypothesis - y_train) ** 2)

    optimizer.zero_grad()
    cost.backward()
    optimizer.step()

    if epoch % 100 == 0:
        print('Epoch {:4d}/{} w1: {:.3f} w2: {:.3f} w3: {:.3f} b: {:.3f} Cost: {:.6f}'.format(
            epoch, nb_epochs, w1.item(), w2.item(), w3.item(), b.item(), cost.item()
        ))

Epoch    0/5000 w1: 0.361 w2: 0.349 w3: 0.367 b: 3.420 Cost: 29661.800781
Epoch  100/5000 w1: 8.021 w2: 7.199 w3: 7.955 b: 148.724 Cost: 517.307983
Epoch  200/5000 w1: 8.342 w2: 6.937 w3: 8.088 b: 167.975 Cost: 10.009558
Epoch  300/5000 w1: 8.552 w2: 6.667 w3: 8.144 b: 170.525 Cost: 0.601720
Epoch  400/5000 w1: 8.725 w2: 6.444 w3: 8.191 b: 170.863 Cost: 0.304983
Epoch  500/5000 w1: 8.867 w2: 6.260 w3: 8.229 b: 170.908 Cost: 0.238107
Epoch  600/5000 w1: 8.984 w2: 6.108 w3: 8.262 b: 170.914 Cost: 0.198604
Epoch  700/5000 w1: 9.079 w2: 5.982 w3: 8.290 b: 170.914 Cost: 0.172262
Epoch  800/5000 w1: 9.158 w2: 5.878 w3: 8.314 b: 170.914 Cost: 0.154346
Epoch  900/5000 w1: 9.222 w2: 5.792 w3: 8.334 b: 170.914 Cost: 0.142090
Epoch 1000/5000 w1: 9.274 w2: 5.721 w3: 8.351 b: 170.914 Cost: 0.133702
Epoch 1100/5000 w1: 9.317 w2: 5.662 w3: 8.366 b: 170.914 Cost: 0.127946
Epoch 1200/5000 w1: 9.352 w2: 5.613 w3: 8.379 b: 170.914 Cost: 0.123995
Epoch 1300/5000 w1: 9.381 w2: 5.572 w3: 8.390 b: 170.914 Co

In [42]:
import torch

# 훈련 데이터
x1 = torch.FloatTensor([73, 93, 89, 96, 73])
x2 = torch.FloatTensor([80, 88, 91, 98, 66])
x3 = torch.FloatTensor([75, 93, 90, 100, 70])
y  = torch.FloatTensor([152, 185, 180, 196, 142])

# 정규화에 쓴 통계
mu    = [x1.mean().item(), x2.mean().item(), x3.mean().item()]
sigma = [x1.std().item(),  x2.std().item(),  x3.std().item()]

# 정규화 버전 가중치를 원래 스케일로 역산하는 함수
def denorm(w_n, b_n):
    w = [w_n[i] / sigma[i] for i in range(3)]
    b = b_n - sum(w_n[i] * mu[i] / sigma[i] for i in range(3))
    return w, b

# ── 세 버전의 가중치 ──
# 1) 비정규화 (이미 원래 스케일)
w_a, b_a = [1.031, 0.513, 0.468], 0.066

# 2) 정규화 (역산 필요)
w_b, b_b = denorm([8.794, 4.999, 9.523], 171.000)

# 3) weight decay (역산 필요)
w_c, b_c = denorm([9.436, 5.342 , 8.558], 170.914)

# ── 예측 함수 ──
def predict(w, b):
    return x1 * w[0] + x2 * w[1] + x3 * w[2] + b

preds = {
    "비정규화":      predict(w_a, b_a),
    "정규화":        predict(w_b, b_b),
    "weight_decay":  predict(w_c, b_c),
}

# ── 행별 예측·오차 출력 ──
header = f"{'실제 y':>7} |" + "".join(f"{name:>22}" for name in preds)
print(header)
print(f"{'':>7} |" + "".join(f"{'예측':>12}{'오차':>10}" for _ in preds))
print("-" * len(header))
for i in range(len(y)):
    row = f"{y[i].item():>7.1f} |"
    for p in preds.values():
        pi = p[i].item()
        row += f"{pi:>12.3f}{pi - y[i].item():>10.3f}"
    print(row)

# ── 전체 MSE ──
print("-" * len(header))
for name, p in preds.items():
    mse = torch.mean((p - y) ** 2).item()
    print(f"{name:>14} MSE: {mse:.6f}")

   실제 y |                  비정규화                   정규화          weight_decay
        |          예측        오차          예측        오차          예측        오차
---------------------------------------------------------------------------
  152.0 |     151.469    -0.531     151.736    -0.264     151.646    -0.354
  185.0 |     184.617    -0.383     184.495    -0.505     184.415    -0.585
  180.0 |     180.628     0.628     180.275     0.275     180.276     0.276
  196.0 |     196.116     0.116     196.248     0.248     196.087     0.087
  142.0 |     141.947    -0.053     142.246     0.246     142.146     0.146
---------------------------------------------------------------------------
          비정규화 MSE: 0.167858
           정규화 MSE: 0.104326
  weight_decay MSE: 0.114410


In [25]:
x_train  =  torch.FloatTensor([[73,  80,  75],
                               [93,  88,  93],
                               [89,  91,  90],
                               [96,  98,  100],
                               [73,  66,  70]])
y_train  =  torch.FloatTensor([[152],  [185],  [180],  [196],  [142]])

W = torch.zeros((3, 1), requires_grad=True)
b = torch.zeros(1, requires_grad=True)

# optimizer 설정
optimizer = optim.SGD([W, b], lr=1e-5)

nb_epochs = 10000
for epoch in range(nb_epochs + 1):

    # H(x) 계산
    # 편향 b는 브로드 캐스팅되어 각 샘플에 더해집니다.
    hypothesis = x_train.matmul(W) + b

    # cost 계산
    cost = torch.mean((hypothesis - y_train) ** 2)

    # cost로 H(x) 개선
    optimizer.zero_grad()
    cost.backward()
    optimizer.step()

    if epoch % 1000 == 0:
      print('Epoch {:4d}/{} W: {} b: {:.3f} Cost: {:.6f}'.format(
        epoch, nb_epochs,
        W.squeeze().detach(),  # [w1, w2, w3] 형태로 출력
        b.item(), cost.item()
      ))

Epoch    0/10000 W: tensor([0.2940, 0.2936, 0.2974]) b: 0.003 Cost: 29661.800781
Epoch 1000/10000 W: tensor([0.7179, 0.6125, 0.6801]) b: 0.009 Cost: 1.079390
Epoch 2000/10000 W: tensor([0.7570, 0.5714, 0.6821]) b: 0.011 Cost: 0.754379
Epoch 3000/10000 W: tensor([0.7879, 0.5405, 0.6821]) b: 0.012 Cost: 0.562653
Epoch 4000/10000 W: tensor([0.8123, 0.5174, 0.6807]) b: 0.013 Cost: 0.448561
Epoch 5000/10000 W: tensor([0.8319, 0.5003, 0.6783]) b: 0.014 Cost: 0.379734
Epoch 6000/10000 W: tensor([0.8477, 0.4877, 0.6751]) b: 0.015 Cost: 0.337358
Epoch 7000/10000 W: tensor([0.8607, 0.4785, 0.6714]) b: 0.016 Cost: 0.310486
Epoch 8000/10000 W: tensor([0.8714, 0.4719, 0.6672]) b: 0.018 Cost: 0.292713
Epoch 9000/10000 W: tensor([0.8804, 0.4673, 0.6628]) b: 0.019 Cost: 0.280343
Epoch 10000/10000 W: tensor([0.8881, 0.4642, 0.6582]) b: 0.020 Cost: 0.271225


In [23]:
# 데이터
x1_train = torch.FloatTensor([[73], [93], [89], [96], [73]])
x2_train = torch.FloatTensor([[80], [88], [91], [98], [66]])
x3_train = torch.FloatTensor([[75], [93], [90], [100], [70]])
y_train = torch.FloatTensor([[152], [185], [180], [196], [142]])

# 정규화 (평균 0, 표준편차 1)
x1_mu, x1_sigma = x1_train.mean(), x1_train.std()
x2_mu, x2_sigma = x2_train.mean(), x2_train.std()
x3_mu, x3_sigma = x3_train.mean(), x3_train.std()
x1_norm = (x1_train - x1_mu) / x1_sigma
x2_norm = (x2_train - x2_mu) / x2_sigma
x3_norm = (x3_train - x3_mu) / x3_sigma

# 모델 초기화
w1 = torch.zeros(1, requires_grad=True)
w2 = torch.zeros(1, requires_grad=True)
w3 = torch.zeros(1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)

optimizer = optim.SGD([w1, w2, w3, b], lr=1e-2)

nb_epochs = 10000
for epoch in range(nb_epochs + 1):
    hypothesis = x1_norm * w1 + x2_norm * w2 + x3_norm * w3 + b
    cost = torch.mean((hypothesis - y_train) ** 2)

    optimizer.zero_grad()
    cost.backward()
    optimizer.step()

    if epoch % 1000 == 0:
        # 정규화 가중치를 원래 스케일로 역산
        w1_o = w1.item() / x1_sigma.item()
        w2_o = w2.item() / x2_sigma.item()
        w3_o = w3.item() / x3_sigma.item()
        b_o = b.item() - (w1.item() * x1_mu.item() / x1_sigma.item()
                          + w2.item() * x2_mu.item() / x2_sigma.item()
                          + w3.item() * x3_mu.item() / x3_sigma.item())
        print('Epoch {:4d}/{} [원래스케일] w1: {:.4f} w2: {:.4f} w3: {:.4f} b: {:.4f} Cost: {:.6f}'.format(
            epoch, nb_epochs, w1_o, w2_o, w3_o, b_o, cost.item()
        ))

print("v1 sigma:", x1_sigma.item(), x2_sigma.item(), x3_sigma.item())
print("v1 x1_norm mean/std:", x1_norm.mean().item(), x1_norm.std().item())

Epoch    0/10000 [원래스케일] w1: 0.0327 w2: 0.0285 w3: 0.0291 b: -4.2502 Cost: 29661.800781
Epoch 1000/10000 [원래스케일] w1: 0.8396 w2: 0.4670 w3: 0.6619 b: 3.6356 Cost: 0.125588
Epoch 2000/10000 [원래스케일] w1: 0.8576 w2: 0.4426 w3: 0.6693 b: 3.5398 Cost: 0.107961
Epoch 3000/10000 [원래스케일] w1: 0.8582 w2: 0.4381 w3: 0.6729 b: 3.5523 Cost: 0.107357
Epoch 4000/10000 [원래스케일] w1: 0.8564 w2: 0.4366 w3: 0.6759 b: 3.5798 Cost: 0.107141
Epoch 5000/10000 [원래스케일] w1: 0.8543 w2: 0.4355 w3: 0.6787 b: 3.6084 Cost: 0.106943
Epoch 6000/10000 [원래스케일] w1: 0.8523 w2: 0.4346 w3: 0.6814 b: 3.6364 Cost: 0.106762
Epoch 7000/10000 [원래스케일] w1: 0.8502 w2: 0.4336 w3: 0.6840 b: 3.6634 Cost: 0.106593
Epoch 8000/10000 [원래스케일] w1: 0.8483 w2: 0.4327 w3: 0.6865 b: 3.6896 Cost: 0.106435
Epoch 9000/10000 [원래스케일] w1: 0.8464 w2: 0.4319 w3: 0.6889 b: 3.7149 Cost: 0.106287
Epoch 10000/10000 [원래스케일] w1: 0.8446 w2: 0.4310 w3: 0.6912 b: 3.7392 Cost: 0.106152
v1 sigma: 11.054410934448242 12.239280700683594 12.621410369873047
v1 x1_norm mea

In [26]:
x_train = torch.FloatTensor([[73, 80, 75],
                             [93, 88, 93],
                             [89, 91, 90],
                             [96, 98, 100],
                             [73, 66, 70]])
y_train = torch.FloatTensor([[152], [185], [180], [196], [142]])

# 정규화 (열 방향: 각 특성별로)
mu = x_train.mean(dim=0)       # shape (3,)
sigma = x_train.std(dim=0)     # shape (3,)
x_norm = (x_train - mu) / sigma

# 모델 초기화
W = torch.zeros((3, 1), requires_grad=True)
b = torch.zeros(1, requires_grad=True)

optimizer = optim.SGD([W, b], lr=1e-2)

nb_epochs = 10000
for epoch in range(nb_epochs + 1):
    hypothesis = x_norm.matmul(W) + b
    cost = torch.mean((hypothesis - y_train) ** 2)

    optimizer.zero_grad()
    cost.backward()
    optimizer.step()

    if epoch % 1000 == 0:
        # 정규화 가중치를 원래 스케일로 역산
        W_flat = W.detach().squeeze()        # shape (3,)
        w_o = W_flat / sigma                  # 원래 스케일 가중치
        b_o = b.item() - (W_flat * mu / sigma).sum().item()
        print('Epoch {:4d}/{} [원래스케일] W: {} b: {:.4f} Cost: {:.6f}'.format(
            epoch, nb_epochs, [round(v, 4) for v in w_o.tolist()], b_o, cost.item()
        ))

print("v2 sigma:", sigma.tolist())
print("v2 x_norm mean/std:", x_norm.mean(dim=0).tolist(), x_norm.std(dim=0).tolist())


Epoch    0/10000 [원래스케일] W: [0.0327, 0.0285, 0.0291] b: -4.2502 Cost: 29661.800781
Epoch 1000/10000 [원래스케일] W: [0.8396, 0.467, 0.6619] b: 3.6356 Cost: 0.125588
Epoch 2000/10000 [원래스케일] W: [0.8576, 0.4426, 0.6693] b: 3.5398 Cost: 0.107961
Epoch 3000/10000 [원래스케일] W: [0.8582, 0.4381, 0.6729] b: 3.5523 Cost: 0.107357
Epoch 4000/10000 [원래스케일] W: [0.8564, 0.4366, 0.6759] b: 3.5798 Cost: 0.107141
Epoch 5000/10000 [원래스케일] W: [0.8543, 0.4355, 0.6787] b: 3.6084 Cost: 0.106943
Epoch 6000/10000 [원래스케일] W: [0.8523, 0.4346, 0.6814] b: 3.6364 Cost: 0.106762
Epoch 7000/10000 [원래스케일] W: [0.8502, 0.4336, 0.684] b: 3.6634 Cost: 0.106593
Epoch 8000/10000 [원래스케일] W: [0.8483, 0.4327, 0.6865] b: 3.6896 Cost: 0.106435
Epoch 9000/10000 [원래스케일] W: [0.8464, 0.4319, 0.6889] b: 3.7148 Cost: 0.106287
Epoch 10000/10000 [원래스케일] W: [0.8446, 0.431, 0.6912] b: 3.7392 Cost: 0.106152
v2 sigma: [11.054410934448242, 12.239280700683594, 12.621410369873047]
v2 x_norm mean/std: [-2.384185791015625e-07, 1.1920928955078125e-07,

# 03-03

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(1)

In [29]:
# 데이터
x_train = torch.FloatTensor([[1], [2], [3]])
y_train = torch.FloatTensor([[2], [4], [6]])

# 모델을 선언 및 초기화. 단순 선형 회귀이므로 input_dim=1, output_dim=1.
model = nn.Linear(1,1)
print(list(model.parameters()))

# optimizer 설정. 경사 하강법 SGD를 사용하고 learning rate를 의미하는 lr은 0.01
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

nb_epochs = 2000
for epoch in range(nb_epochs+1):

    # H(x) 계산
    prediction = model(x_train)

    # cost 계산
    cost = F.mse_loss(prediction, y_train) # <== 파이토치에서 제공하는 평균 제곱 오차 함수

    # cost로 H(x) 개선하는 부분
    # gradient를 0으로 초기화
    optimizer.zero_grad()
    # 비용 함수를 미분하여 gradient 계산
    cost.backward() # backward 연산
    # W와 b를 업데이트
    optimizer.step()

    if epoch % 100 == 0:
    # 100번마다 로그 출력
      print('Epoch {:4d}/{} Cost: {:.6f}'.format(
          epoch, nb_epochs, cost.item()
      ))

# 임의의 입력 4를 선언
new_var =  torch.FloatTensor([[4.0]])
# 입력한 값 4에 대해서 예측값 y를 리턴받아서 pred_y에 저장
pred_y = model(new_var) # forward 연산
# y = 2x 이므로 입력이 4라면 y가 8에 가까운 값이 나와야 제대로 학습이 된 것
print("훈련 후 입력이 4일 때의 예측값 :", pred_y)

print(list(model.parameters()))


[Parameter containing:
tensor([[-0.9414]], requires_grad=True), Parameter containing:
tensor([0.5997], requires_grad=True)]
Epoch    0/2000 Cost: 33.679783
Epoch  100/2000 Cost: 0.223159
Epoch  200/2000 Cost: 0.137899
Epoch  300/2000 Cost: 0.085213
Epoch  400/2000 Cost: 0.052656
Epoch  500/2000 Cost: 0.032539
Epoch  600/2000 Cost: 0.020107
Epoch  700/2000 Cost: 0.012425
Epoch  800/2000 Cost: 0.007678
Epoch  900/2000 Cost: 0.004744
Epoch 1000/2000 Cost: 0.002932
Epoch 1100/2000 Cost: 0.001812
Epoch 1200/2000 Cost: 0.001119
Epoch 1300/2000 Cost: 0.000692
Epoch 1400/2000 Cost: 0.000427
Epoch 1500/2000 Cost: 0.000264
Epoch 1600/2000 Cost: 0.000163
Epoch 1700/2000 Cost: 0.000101
Epoch 1800/2000 Cost: 0.000062
Epoch 1900/2000 Cost: 0.000039
Epoch 2000/2000 Cost: 0.000024
훈련 후 입력이 4일 때의 예측값 : tensor([[7.9902]], grad_fn=<AddmmBackward0>)
[Parameter containing:
tensor([[1.9943]], requires_grad=True), Parameter containing:
tensor([0.0128], requires_grad=True)]


In [33]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(1)

In [34]:
# 데이터
x_train = torch.FloatTensor([[73, 80, 75],
                             [93, 88, 93],
                             [89, 91, 90],
                             [96, 98, 100],
                             [73, 66, 70]])
y_train = torch.FloatTensor([[152], [185], [180], [196], [142]])

# 모델을 선언 및 초기화. 단순 선형 회귀이므로 input_dim=1, output_dim=1.
model = nn.Linear(3,1)
print(list(model.parameters()))

# optimizer 설정. 경사 하강법 SGD를 사용하고 learning rate를 의미하는 lr은 0.01
optimizer = torch.optim.SGD(model.parameters(), lr=1e-5)

nb_epochs = 2000
for epoch in range(nb_epochs+1):

    # H(x) 계산
    prediction = model(x_train)

    # cost 계산
    cost = F.mse_loss(prediction, y_train) # <== 파이토치에서 제공하는 평균 제곱 오차 함수

    # cost로 H(x) 개선하는 부분
    # gradient를 0으로 초기화
    optimizer.zero_grad()
    # 비용 함수를 미분하여 gradient 계산
    cost.backward() # backward 연산
    # W와 b를 업데이트
    optimizer.step()

    if epoch % 100 == 0:
    # 100번마다 로그 출력
      print('Epoch {:4d}/{} Cost: {:.6f}'.format(
          epoch, nb_epochs, cost.item()
      ))

# 임의의 입력을 선언
new_var =  torch.FloatTensor([[73, 80, 75]])
# 입력한 값 4에 대해서 예측값 y를 리턴받아서 pred_y에 저장
pred_y = model(new_var) # forward 연산
# y = 2x 이므로 입력이 4라면 y가 8에 가까운 값이 나와야 제대로 학습이 된 것
print("훈련 후 입력의 예측값 :", pred_y)

print(list(model.parameters()))


[Parameter containing:
tensor([[ 0.2975, -0.2548, -0.1119]], requires_grad=True), Parameter containing:
tensor([0.2710], requires_grad=True)]
Epoch    0/2000 Cost: 31667.597656
Epoch  100/2000 Cost: 0.225993
Epoch  200/2000 Cost: 0.223911
Epoch  300/2000 Cost: 0.221941
Epoch  400/2000 Cost: 0.220059
Epoch  500/2000 Cost: 0.218271
Epoch  600/2000 Cost: 0.216575
Epoch  700/2000 Cost: 0.214950
Epoch  800/2000 Cost: 0.213413
Epoch  900/2000 Cost: 0.211952
Epoch 1000/2000 Cost: 0.210560
Epoch 1100/2000 Cost: 0.209232
Epoch 1200/2000 Cost: 0.207967
Epoch 1300/2000 Cost: 0.206761
Epoch 1400/2000 Cost: 0.205619
Epoch 1500/2000 Cost: 0.204522
Epoch 1600/2000 Cost: 0.203484
Epoch 1700/2000 Cost: 0.202485
Epoch 1800/2000 Cost: 0.201542
Epoch 1900/2000 Cost: 0.200635
Epoch 2000/2000 Cost: 0.199769
훈련 후 입력의 예측값 : tensor([[151.2305]], grad_fn=<AddmmBackward0>)
[Parameter containing:
tensor([[0.9778, 0.4539, 0.5768]], requires_grad=True), Parameter containing:
tensor([0.2802], requires_grad=True)]


# 03-04

In [42]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import TensorDataset # 텐서데이터셋
from torch.utils.data import DataLoader # 데이터로더

torch.manual_seed(1)

x_train  =  torch.FloatTensor([[73,  80,  75],
                               [93,  88,  93],
                               [89,  91,  90],
                               [96,  98,  100],
                               [73,  66,  70]])
y_train  =  torch.FloatTensor([[152],  [185],  [180],  [196],  [142]])

dataset = TensorDataset(x_train, y_train)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

model = nn.Linear(3,1)
optimizer = torch.optim.SGD(model.parameters(), lr=1e-5)

nb_epochs = 20
for epoch in range(nb_epochs + 1):
  for batch_idx, samples in enumerate(dataloader):
    x_train, y_train = samples
    # H(x) 계산
    prediction = model(x_train)

    # cost 계산
    cost = F.mse_loss(prediction, y_train)

    # cost로 H(x) 계산
    optimizer.zero_grad()
    cost.backward()
    optimizer.step()

    print('Epoch {:4d}/{} Batch {}/{} Cost: {:.6f}'.format(
        epoch, nb_epochs, batch_idx+1, len(dataloader),
        cost.item()
        ))

# 임의의 입력 [73, 80, 75]를 선언
new_var =  torch.FloatTensor([[73, 80, 75]])
# 입력한 값 [73, 80, 75]에 대해서 예측값 y를 리턴받아서 pred_y에 저장
pred_y = model(new_var)
print("훈련 후 입력이 73, 80, 75일 때의 예측값 :", pred_y)


Epoch    0/20 Batch 1/3 Cost: 23071.781250
Epoch    0/20 Batch 2/3 Cost: 17581.359375
Epoch    0/20 Batch 3/3 Cost: 3703.553467
Epoch    1/20 Batch 1/3 Cost: 857.131836
Epoch    1/20 Batch 2/3 Cost: 194.912628
Epoch    1/20 Batch 3/3 Cost: 103.150658
Epoch    2/20 Batch 1/3 Cost: 16.460945
Epoch    2/20 Batch 2/3 Cost: 10.970690
Epoch    2/20 Batch 3/3 Cost: 2.953053
Epoch    3/20 Batch 1/3 Cost: 1.246350
Epoch    3/20 Batch 2/3 Cost: 0.095024
Epoch    3/20 Batch 3/3 Cost: 0.104377
Epoch    4/20 Batch 1/3 Cost: 0.678422
Epoch    4/20 Batch 2/3 Cost: 0.123370
Epoch    4/20 Batch 3/3 Cost: 0.118080
Epoch    5/20 Batch 1/3 Cost: 0.094797
Epoch    5/20 Batch 2/3 Cost: 0.531496
Epoch    5/20 Batch 3/3 Cost: 0.011198
Epoch    6/20 Batch 1/3 Cost: 0.219984
Epoch    6/20 Batch 2/3 Cost: 0.042412
Epoch    6/20 Batch 3/3 Cost: 0.917457
Epoch    7/20 Batch 1/3 Cost: 0.213805
Epoch    7/20 Batch 2/3 Cost: 0.640249
Epoch    7/20 Batch 3/3 Cost: 0.005267
Epoch    8/20 Batch 1/3 Cost: 0.454904
Epoch 

In [46]:
# Dataset 상속
class CustomDataset(Dataset):
  def __init__(self):
    self.x_data = [[73, 80, 75],
                   [93, 88, 93],
                   [89, 91, 90],
                   [96, 98, 100],
                   [73, 66, 70]]
    self.y_data = [[152], [185], [180], [196], [142]]

  # 총 데이터의 개수를 리턴
  def __len__(self):
    return len(self.x_data)

  # 인덱스를 입력받아 그에 맵핑되는 입출력 데이터를 파이토치의 Tensor 형태로 리턴
  def __getitem__(self, idx):
    x = torch.FloatTensor(self.x_data[idx])
    y = torch.FloatTensor(self.y_data[idx])
    return x, y

dataset = CustomDataset()
print(len(dataset))
print(dataset[0])

5
(tensor([73., 80., 75.]), tensor([152.]))


# 03-05

In [69]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(1)

train_x = torch.FloatTensor([[0, 0,], [1, 0], [0, 1], [1, 1]])
train_y = torch.FloatTensor([[0], [1], [1], [0]])

model = nn.Linear(2, 1)

optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

nb_epochs = 3000
for epoch in range(nb_epochs + 1):
  prediction = model(train_x)

  cost = F.mse_loss(prediction, train_y)

  optimizer.zero_grad()
  cost.backward()
  optimizer.step()

  if epoch % 1000 == 0:
    param_str = ', '.join(
      f'{name}={[round(v, 4) for v in param.detach().flatten().tolist()]}'
      for name, param in model.named_parameters()
  )
    print(f'Epoch {epoch}/{nb_epochs} Parameters: {param_str} Cost: {cost.item():6f}')



Epoch 0/3000 Parameters: weight=[0.3686, -0.3044], bias=[-0.1249] Cost: 0.680809
Epoch 1000/3000 Parameters: weight=[0.0124, 0.008], bias=[0.4879] Cost: 0.250058
Epoch 2000/3000 Parameters: weight=[0.0005, 0.0004], bias=[0.4995] Cost: 0.250000
Epoch 3000/3000 Parameters: weight=[0.0, 0.0], bias=[0.5] Cost: 0.250000


In [65]:
test_x = torch.FloatTensor([[0, 0,], [1, 0], [0, 1], [1, 1]])
pred_y = model(test_x)
print(pred_y)

tensor([[0.5000],
        [0.5000],
        [0.5000],
        [0.5000]], grad_fn=<AddmmBackward0>)


In [71]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(2)

train_x = torch.FloatTensor([[0, 0], [1, 0], [0, 1], [1, 1]])
train_y = torch.FloatTensor([[0], [1], [1], [0]])

# 은닉층 모델: 2 → 2 → 1, 비선형 활성화(Sigmoid) 포함
model = nn.Sequential(
    nn.Linear(2, 2),
    nn.Sigmoid(),
    nn.Linear(2, 1),
    nn.Sigmoid()
)

optimizer = torch.optim.SGD(model.parameters(), lr=1.0)

nb_epochs = 10000
for epoch in range(nb_epochs + 1):
    prediction = model(train_x)
    cost = F.mse_loss(prediction, train_y)

    optimizer.zero_grad()
    cost.backward()
    optimizer.step()

    if epoch % 1000 == 0:
        param_str = ', '.join(
            f'{name}={[round(v, 4) for v in param.detach().flatten().tolist()]}'
            for name, param in model.named_parameters()
        )
        print(f'Epoch {epoch}/{nb_epochs} Parameters: {param_str} Cost: {cost.item():6f}')

Epoch 0/10000 Parameters: 0.weight=[0.162, -0.1685, 0.1921, -0.0381], 0.bias=[0.3016, 0.1648], 2.weight=[-0.0667, -0.5574], 2.bias=[0.1868] Cost: 0.252637
Epoch 1000/10000 Parameters: 0.weight=[0.1677, -0.1726, 0.2049, -0.1291], 0.bias=[0.3047, 0.2151], 2.weight=[-0.0265, -0.4873], 2.bias=[0.2893] Cost: 0.249988
Epoch 2000/10000 Parameters: 0.weight=[0.1737, -0.1752, 0.4117, -0.2924], 0.bias=[0.3096, 0.4054], 2.weight=[-0.0056, -0.5099], 2.bias=[0.3145] Cost: 0.249854
Epoch 3000/10000 Parameters: 0.weight=[2.0914, -2.4329, 4.2587, -4.0293], 0.bias=[-0.9782, 2.5749], 2.weight=[3.4952, -3.8895], 2.bias=[1.8544] Cost: 0.085044
Epoch 4000/10000 Parameters: 0.weight=[4.4928, -4.7408, 5.5507, -5.5315], 0.bias=[-2.4393, 2.9033], 2.weight=[7.4076, -6.8931], 2.bias=[3.1513] Cost: 0.003288
Epoch 5000/10000 Parameters: 0.weight=[4.8527, -5.0925, 5.8294, -5.7956], 0.bias=[-2.6314, 2.9998], 2.weight=[8.1564, -7.6345], 2.bias=[3.5227] Cost: 0.001511
Epoch 6000/10000 Parameters: 0.weight=[5.0439, -5.

In [72]:
# 학습 후 예측 확인
test_x = torch.FloatTensor([[0, 0], [1, 0], [0, 1], [1, 1]])
with torch.no_grad():
    pred_y = model(test_x)
    print('\n예측값:')
    print(pred_y.squeeze().tolist())
    print('이진 분류 결과:', (pred_y > 0.5).float().squeeze().tolist())
    print('정답:', train_y.squeeze().tolist())


예측값:
[0.019661840051412582, 0.9813298583030701, 0.9776681661605835, 0.01754765957593918]
이진 분류 결과: [0.0, 1.0, 1.0, 0.0]
정답: [0.0, 1.0, 1.0, 0.0]
